# 01 - Auditoria dos dados
Objetivo: conferir o que entrou antes de modelar. Base oficial do LAPAE: contagem mensal de exames por origem (formato largo na planilha).

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from src.config import *
from src.dados import *
from src.variaveis import *
np.random.seed(SEMENTE)
PROCESSADO.mkdir(parents=True, exist_ok=True)
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)

In [2]:
bruto = carregar_bruto()
assert bruto is not None, "Coloque a planilha em dados/bruto/ (veja dados/bruto/LEIAME.md)"
bruto = bruto.drop(columns="arquivo_origem")
base, log, total_entrada = preparar_base_mensal(bruto)
print("Entrada:", bruto.shape, "| colunas:", list(bruto.columns)[:3], "...")
display(log)

Entrada: (6, 15) | colunas: ['Unnamed: 0', 'Agosto(2025)', 'Setembro(2025)'] ...


,etapa,linhas_antes,linhas_depois
0,descartar mês ou origem ausentes,84,84


## Verificações de esquema e de totais

In [3]:
esquema = validar_esquema(base); display(esquema)
totais = validar_totais(total_entrada, base); print(totais)
assert esquema["ok"].all() and totais["totais_conferem"]

,verificação,ok,detalhe
0,colunas obrigatórias presentes,True,
1,mês é do tipo data,True,
2,mês é sempre o 1º dia do mês,True,
3,exames é numérico,True,
4,exames não é negativo,True,
5,exames é inteiro (quando presente),True,
6,"chave (origem, mês) é única",True,0 linhas repetidas
7,origem sem espaços nas pontas,True,
8,origem sem grafias duplicadas,True,6 grafias para 6 origens
9,"meses consecutivos, sem buracos",True,14 de 14 meses


{'total_de_entrada': 20913.0, 'soma_base_mensal': 20913.0, 'totais_conferem': True, 'celulas_grade': 84, 'celulas_ausentes': 0, 'origens': 6, 'meses': 14}


## Perfil por origem

In [4]:
perfil = base.groupby(COL_ORIGEM)[COL_ALVO].agg(["sum","mean","min","max","std"]).round(1)
perfil["participacao_%"] = (100*perfil["sum"]/perfil["sum"].sum()).round(1)
perfil["cv"] = (perfil["std"]/perfil["mean"]).round(2)
display(perfil.sort_values("sum", ascending=False))
mensal = base.pivot(index=COL_MES, columns=COL_ORIGEM, values=COL_ALVO); mensal["TOTAL"] = mensal.sum(axis=1)
display(mensal)
print("Média mensal do total:", round(mensal["TOTAL"].mean(),1), "| capacidade informada:", CAPACIDADE_MEDIA)

,sum,mean,min,max,std,participacao_%,cv
origem,,,,,,,
SUS,8794,628.1,374,886,143.2,42.1,0.23
MAURILIO DE ALMEIDA,6813,486.6,262,728,120.2,32.6,0.25
ROSEANE DORE,3841,274.4,203,344,43.5,18.4,0.16
SAO LUCAS,1228,87.7,24,132,30.6,5.9,0.35
VIVER MAIS,153,10.9,2,41,10.0,0.7,0.92
PARTICULARES,84,6.0,1,18,5.4,0.4,0.90


origem,MAURILIO DE ALMEIDA,PARTICULARES,ROSEANE DORE,SAO LUCAS,SUS,VIVER MAIS,TOTAL
mes,,,,,,,
2025-08-01,262,4,314,24,552,9,1165
2025-09-01,602,1,292,82,483,10,1470
2025-10-01,728,5,303,85,809,41,1971
2025-11-01,417,4,228,70,653,20,1392
2025-12-01,324,3,203,54,680,9,1273
2026-01-01,528,2,275,127,374,5,1311
2026-02-01,433,5,253,81,804,6,1582
2026-03-01,573,6,340,123,689,3,1734
2026-04-01,417,3,250,104,886,2,1662


Média mensal do total: 1493.8 | capacidade informada: 1850


In [5]:
fig, ax = plt.subplots(figsize=(9,3.6))
mensal.drop(columns="TOTAL").plot(ax=ax, marker="o", lw=1.3)
ax.set_ylabel("exames por mês"); ax.set_title("Exames por origem e mês"); ax.legend(fontsize=7, ncol=3)
fig.tight_layout(); fig.savefig(FIG/"serie_por_origem.png", dpi=150)
fig, ax = plt.subplots(figsize=(9,3))
ax.bar(mensal.index.strftime("%b/%y"), mensal["TOTAL"], color="#0B3954"); ax.axhline(CAPACIDADE_MEDIA, color="#C86A12", ls="--", label="capacidade informada (1.850)")
ax.set_title("Total mensal de exames"); ax.legend(); plt.xticks(rotation=45); fig.tight_layout(); fig.savefig(FIG/"total_mensal.png", dpi=150)

## Observações
- 14 meses (ago/2025 a set/2026): cada mês do ano aparece no máximo 2 vezes (ago e set) e outubro a dezembro aparecem 1 vez.
- A base oficial não traz município; a série é a origem do exame.
- Nenhuma célula ausente; ausência não foi convertida em zero em nenhuma etapa.